# Alert → Library Risk (LR) / Library Control (L1C) / Control Instance (1C) — management story
Same storyline as the alert → regulation sheet, built **only from the audited Run B outputs**:
1. Universe and scope → 2. What we did (combinations → candidates → links) → 3. Why not golden (grain)
→ 4. Can we trust the judge? (probes, rejected sample) → 5. What the judge says about the links
→ 6. Why links get disputed → 7. Score cut-off → 8. One-page summary.

Every number is calculated here from the files below; column names were confirmed by the readiness checks.
Output: printed story + `helios_story_numbers.xlsx` (small, < 100 KB).

In [ ]:
################################################################################
# CELL 1 - FILES AND CONFIRMED COLUMN NAMES
################################################################################
import os, math
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

DATA = "../data"
RAPID2_FILE   = f"{DATA}/Rapid2_RESDEV_US_2020_2025.parquet"
GOLDEN_FILE   = f"{DATA}/golden_dataset_US_exploded_with_control_mapping_03sep26.parquet"
RISK_FILE     = f"{DATA}/Helios/US Risk.parquet"
CONTROL_FILE  = f"{DATA}/Helios/US_L1_Control.parquet"
RUN_B_FOLDER  = "output/helios_linkage_llm_RT_method2"
AUDIT_B_FILE  = "output/helios_audit_B/audit_labelled.parquet"
OUTPUT_FILE   = "helios_story_numbers.xlsx"

RAPID2_ALERT = "RECORD_ID"
GOLDEN_ALERT = "Record ID"
GOLDEN_TARGET = {"LR":  "Regulation Summary Library Risk ID Number",
                 "L1C": "Regulation Summary L1 Library Control ID Number",
                 "1C":  "L1 Control ID"}

RISK_LIBRARY_ID, RISK_LIBRARY_STATUS, RISK_INSTANCE_STATUS = "library_risk_id", "library_risk_status", "risk_status"
CONTROL_LIBRARY_ID, CONTROL_LIBRARY_STATUS = "l1_library_control_id", "l1_library_control_status"
INSTANCE_ID, INSTANCE_STATUS = "name", "l1_control_status"

# LR universe definition - both are printed; this one drives the story
LR_LIVE_RULE = "published_with_approved_instance"     # or "published"

LEVELS = {   # level code -> (links file, target column in links, audit level value)
    "LR":  ("links_risk.parquet",     "LR",  "risk"),
    "L1C": ("links_control.parquet",  "LIC", "control"),
    "1C":  ("links_instance.parquet", "IC",  "instance"),
}
LEVEL_NAMES = {"LR": "Library risk", "L1C": "Library control", "1C": "Control instance"}

In [ ]:
################################################################################
# CELL 2 - HELPERS
################################################################################
def read(path):
    return pq.read_table(path).to_pandas(ignore_metadata=True)      # avoids the 'dbdate' error


def clean(series):
    return series.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)


def upper_label(series):
    return series.astype(str).str.strip().str.upper().str.replace(" ", "_")


def wilson(successes, total, z=1.96):
    if total == 0:
        return (float("nan"), float("nan"))
    p = successes / total
    centre = (p + z * z / (2 * total)) / (1 + z * z / total)
    half = z * math.sqrt(p * (1 - p) / total + z * z / (4 * total * total)) / (1 + z * z / total)
    return (max(0.0, centre - half), min(1.0, centre + half))


def pct(part, whole, digits=1):
    return round(100 * part / whole, digits) if whole else float("nan")


def ci_text(successes, total):
    low, high = wilson(successes, total)
    return f"{100 * low:.1f}% - {100 * high:.1f}%" if total else "n/a"

In [ ]:
################################################################################
# CELL 3 - SECTION 1: UNIVERSE AND SCOPE
################################################################################
rapid2 = read(RAPID2_FILE)
golden = read(GOLDEN_FILE)
risk = read(RISK_FILE)
control = read(CONTROL_FILE)
audit = read(AUDIT_B_FILE)
audit["level_code"] = audit["level"].astype(str).str.lower().map({v[2]: k for k, v in LEVELS.items()})
audit["label"] = upper_label(audit["audit_label"])
audit["bucket_clean"] = upper_label(audit["bucket"])

all_alerts = set(clean(rapid2[RAPID2_ALERT]))
golden_alerts = set(clean(golden[GOLDEN_ALERT]))
links_by_level = {code: read(os.path.join(RUN_B_FOLDER, spec[0])) for code, spec in LEVELS.items()}
run_alerts = set().union(*[set(clean(t["RID"])) for t in links_by_level.values()])
scope_alerts = golden_alerts & all_alerts
assert len(golden_alerts) == len(scope_alerts), "some golden alerts are not in RAPID2 - check before presenting"

# LR universe
risk_published = risk[risk[RISK_LIBRARY_STATUS].astype(str).str.upper() == "PUBLISHED"]
lr_published = set(clean(risk_published[RISK_LIBRARY_ID]))
lr_with_approved = set(clean(risk_published[risk_published[RISK_INSTANCE_STATUS].astype(str).str.upper() == "APPROVED"][RISK_LIBRARY_ID]))
universe = {"LR": lr_with_approved if LR_LIVE_RULE == "published_with_approved_instance" else lr_published}
# L1C universe: published library controls; 1C universe: assessed instances
universe["L1C"] = set(clean(control[control[CONTROL_LIBRARY_STATUS].astype(str).str.upper() == "PUBLISHED"][CONTROL_LIBRARY_ID]))
universe["1C"] = set(clean(control[control[INSTANCE_STATUS].astype(str).str.upper() == "ASSESSED"][INSTANCE_ID]))

section_1 = pd.DataFrame([
    ("RAPID2 US alerts 2020-2025", len(all_alerts)),
    ("Alerts in scope (golden subset from business)", len(scope_alerts)),
    ("Alerts in Run B outputs", len(run_alerts)),
    ("Run B alerts that are in scope", len(run_alerts & scope_alerts)),
    ("Helios library risks (all, US)", int(clean(risk[RISK_LIBRARY_ID]).nunique())),
    ("  - Published", len(lr_published)),
    ("  - Published with >=1 Approved US risk instance", len(lr_with_approved)),
    (f"  => LR universe used ({LR_LIVE_RULE})", len(universe["LR"])),
    ("Helios library controls (all, US)", int(clean(control[CONTROL_LIBRARY_ID]).nunique())),
    ("  => L1C universe used (Published)", len(universe["L1C"])),
    ("Helios control instances (all, US)", int(clean(control[INSTANCE_ID]).nunique())),
    ("  => 1C universe used (Assessed)", len(universe["1C"])),
], columns=["item", "value"])
print("SECTION 1 - UNIVERSE AND SCOPE"); print(section_1.to_string(index=False))

In [ ]:
################################################################################
# CELL 4 - SECTION 2: WHAT WE DID (combinations -> candidates -> links), per level
################################################################################
linked_by_level, candidates_by_level, rows_2 = {}, {}, []
for code, (file_name, target_column, _) in LEVELS.items():
    links = links_by_level[code].copy()
    links["alert"], links["target"] = clean(links["RID"]), clean(links[target_column])
    links = links[links.alert.isin(scope_alerts)]
    candidates = links.drop_duplicates(["alert", "target"])
    linked = candidates[upper_label(candidates["final_decision"]) == "LINKED"]
    linked_by_level[code] = linked
    candidates_by_level[code] = candidates
    provenance = upper_label(linked["provenance"]).value_counts() if "provenance" in linked else pd.Series(dtype=int)
    rows_2.append({
        "level": LEVEL_NAMES[code],
        "total possible combinations (alerts x universe)": len(scope_alerts) * len(universe[code]),
        "candidate pairs judged by model": len(candidates),
        "alerts with >=1 candidate": candidates.alert.nunique(),
        "LINKED pairs": len(linked),
        "link rate among candidates %": pct(len(linked), len(candidates)),
        "alerts with >=1 link": linked.alert.nunique(),
        f"distinct items linked": linked.target.nunique(),
        "share of universe linked %": pct(len(set(linked.target) & universe[code]), len(universe[code])),
        "links outside live universe": len(set(linked.target) - universe[code]),
        "links found via structure only": int(provenance.filter(like="STRUCTURAL").sum()),
        "links found via text search only": int(provenance.filter(like="TEXT").sum()),
        "links found via both": int(provenance.get("BOTH", 0)),
    })
section_2 = pd.DataFrame(rows_2).set_index("level").astype(object).T
print("\nSECTION 2 - WHAT WE DID"); print(section_2.to_string())

In [ ]:
################################################################################
# CELL 5 - SECTION 3: WHY WE DON'T USE THE GOLDEN DATA AT THIS LEVEL (grain)
################################################################################
rows_3 = []
golden_rows_per_alert = golden.groupby(clean(golden[GOLDEN_ALERT])).size()
for code, golden_column in GOLDEN_TARGET.items():
    golden_pairs = set(zip(clean(golden[GOLDEN_ALERT]), clean(golden[golden_column])))
    golden_pairs = {p for p in golden_pairs if p[1] not in ("", "nan", "None")}
    linked = linked_by_level[code]
    linked_pairs = set(zip(linked.alert, linked.target))
    per_alert = pd.Series([a for a, _ in golden_pairs]).value_counts()
    labels = audit[(audit.level_code == code) & (audit.bucket_clean == "LINKED")]
    labels = labels.assign(pair=list(zip(clean(labels["RID"]), clean(labels["TARGET"]))))
    in_golden = labels[labels.pair.isin(golden_pairs)]
    not_golden = labels[~labels.pair.isin(golden_pairs)]
    rows_3.append({
        "level": LEVEL_NAMES[code],
        "golden alert-item pairs": len(golden_pairs),
        "golden items per alert (median)": float(per_alert.median()) if len(per_alert) else 0,
        "golden items per alert (max)": int(per_alert.max()) if len(per_alert) else 0,
        "model links also in golden": len(linked_pairs & golden_pairs),
        "golden pairs the model linked %": pct(len(linked_pairs & golden_pairs), len(golden_pairs)),
        "judge CONFIRMED % - links in golden": pct((in_golden.label == "CONFIRMED").sum(), len(in_golden)),
        "judge CONFIRMED % - links not in golden": pct((not_golden.label == "CONFIRMED").sum(), len(not_golden)),
    })
section_3 = pd.DataFrame(rows_3).set_index("level").T
print("\nSECTION 3 - GOLDEN DATA AT LR / L1C / 1C GRAIN")
print(f"golden rows {len(golden):,} for {len(golden_alerts)} alerts (median {golden_rows_per_alert.median():,.0f} rows per alert): "
      "every control on the regulation summary is copied to the alert, so it is not alert-level truth")
print(section_3.to_string())

In [ ]:
################################################################################
# CELL 6 - SECTION 4: CAN WE TRUST THE JUDGE? (planted fakes + rejected sample)
################################################################################
rows_4 = []
for code in LEVELS:
    level_audit = audit[audit.level_code == code]
    probes = level_audit[level_audit.bucket_clean == "PROBE"]
    caught = int((probes.label == "PROBE_CAUGHT").sum())
    rejected = level_audit[level_audit.bucket_clean == "REJECTED_SAMPLE"]
    possible_miss = int((rejected.label == "POSSIBLE_MISS").sum())
    not_linked_total = len(candidates_by_level[code]) - len(linked_by_level[code])
    miss_rate = possible_miss / len(rejected) if len(rejected) else float("nan")
    miss_low, miss_high = wilson(possible_miss, len(rejected))
    rows_4.append({
        "level": LEVEL_NAMES[code],
        "planted wrong links": len(probes),
        "caught by judge": caught,
        "probe catch rate %": pct(caught, len(probes)),
        "probe catch rate 95% CI": ci_text(caught, len(probes)),
        "rejected pairs sampled": len(rejected),
        "judge agrees rejection was right": int((rejected.label == "CORRECT_REJECTION").sum()),
        "possible misses (judge says should link)": possible_miss,
        "possible miss rate %": pct(possible_miss, len(rejected), 2),
        "possible miss rate 95% CI": ci_text(possible_miss, len(rejected)),
        "rejected pairs in total": not_linked_total,
        "estimated missed links (rate x rejected)": f"{miss_rate * not_linked_total:,.0f} "
                                                    f"({miss_low * not_linked_total:,.0f} - {miss_high * not_linked_total:,.0f})",
    })
section_4 = pd.DataFrame(rows_4).set_index("level").T
print("\nSECTION 4 - JUDGE RELIABILITY"); print(section_4.to_string())

In [ ]:
################################################################################
# CELL 7 - SECTION 5 + 6: WHAT THE JUDGE SAYS ABOUT THE LINKS, AND WHY LINKS GET DISPUTED
################################################################################
rows_5, rows_6, linked_audit_by_level = [], [], {}
for code in LEVELS:
    judged = audit[(audit.level_code == code) & (audit.bucket_clean == "LINKED")].copy()
    judged["alert"] = clean(judged["RID"])
    linked_audit_by_level[code] = judged
    total = len(judged)
    confirmed = int((judged.label == "CONFIRMED").sum())
    rows_5.append({
        "level": LEVEL_NAMES[code],
        "LINKED pairs": len(linked_by_level[code]),
        "audited by judge": total,
        "audit coverage %": pct(total, len(linked_by_level[code])),
        "CONFIRMED": confirmed,
        "DISPUTED": int((judged.label == "DISPUTED").sum()),
        "NEEDS_SME": int((judged.label == "NEEDS_SME").sum()),
        "CONFIRMED %": pct(confirmed, total),
        "CONFIRMED 95% CI": ci_text(confirmed, total),
        "1 in N links confirmed": round(total / confirmed, 1) if confirmed else float("nan"),
        "alerts with >=1 link": judged.alert.nunique(),
        "alerts with >=1 CONFIRMED link": judged[judged.label == "CONFIRMED"].alert.nunique(),
        "distinct items CONFIRMED": clean(judged[judged.label == "CONFIRMED"]["TARGET"]).nunique(),
    })
    disputed = judged[judged.label == "DISPUTED"]
    reasons = upper_label(disputed["audit_error_type"]).value_counts()
    for reason, count in reasons.items():
        rows_6.append({"level": LEVEL_NAMES[code], "dispute reason": reason, "links": int(count),
                       "% of disputed": pct(count, len(disputed))})
section_5 = pd.DataFrame(rows_5).set_index("level").T
section_6 = pd.DataFrame(rows_6)
print("\nSECTION 5 - JUDGE VERDICTS ON THE LINKS"); print(section_5.to_string())
print("\nSECTION 6 - WHY LINKS ARE DISPUTED"); print(section_6.to_string(index=False))

In [ ]:
################################################################################
# CELL 8 - SECTION 7: SCORE CUT-OFF (keep links with link_score >= row)
################################################################################
cutoff_tables = {}
for code in LEVELS:
    judged = linked_audit_by_level[code].copy()
    judged["score"] = pd.to_numeric(judged["link_score"], errors="coerce")
    table = pd.crosstab(judged.score, judged.label).reindex(columns=["CONFIRMED", "DISPUTED", "NEEDS_SME"], fill_value=0)
    table["total"] = table.sum(axis=1)
    kept = table[::-1].cumsum()[::-1]
    table["keep >= score: CONFIRMED kept %"] = (100 * kept.CONFIRMED / max(table.CONFIRMED.sum(), 1)).round(0)
    table["keep >= score: volume kept %"] = (100 * kept.total / max(table.total.sum(), 1)).round(0)
    table["keep >= score: CONFIRMED rate %"] = (100 * kept.CONFIRMED / kept.total).round(0)
    table.index = table.index.map(lambda s: f"{s:g}")
    totals = table[["CONFIRMED", "DISPUTED", "NEEDS_SME", "total"]].sum()
    table.loc["Grand total"] = list(totals) + ["", "", ""]
    cutoff_tables[code] = table
    print(f"\nSECTION 7 - SCORE CUT-OFF: {LEVEL_NAMES[code]}"); print(table.to_string())

In [ ]:
################################################################################
# CELL 9 - SECTION 8: ONE-PAGE SUMMARY + SAVE
################################################################################
summary_rows = []
for code in LEVELS:
    s2, s4, s5 = section_2[LEVEL_NAMES[code]], section_4[LEVEL_NAMES[code]], section_5[LEVEL_NAMES[code]]
    summary_rows.append({
        "level": LEVEL_NAMES[code],
        "universe": len(universe[code]),
        "possible pairs": s2["total possible combinations (alerts x universe)"],
        "candidates judged": s2["candidate pairs judged by model"],
        "linked": s2["LINKED pairs"],
        "confirmed by judge": s5["CONFIRMED"],
        "confirmed %": s5["CONFIRMED %"],
        "confirmed 95% CI": s5["CONFIRMED 95% CI"],
        "alerts with confirmed link": s5["alerts with >=1 CONFIRMED link"],
        "probes caught": f"{s4['caught by judge']}/{s4['planted wrong links']}",
        "possible miss rate %": s4["possible miss rate %"],
    })
summary = pd.DataFrame(summary_rows).set_index("level").astype(object)
print("\nSECTION 8 - ONE-PAGE SUMMARY"); print(summary.T.to_string())

with pd.ExcelWriter(OUTPUT_FILE) as writer:
    summary.T.to_excel(writer, sheet_name="Summary")
    section_1.to_excel(writer, sheet_name="1 Universe", index=False)
    section_2.to_excel(writer, sheet_name="2 What we did")
    section_3.to_excel(writer, sheet_name="3 Golden grain")
    section_4.to_excel(writer, sheet_name="4 Judge reliability")
    section_5.to_excel(writer, sheet_name="5 Judge verdicts")
    section_6.to_excel(writer, sheet_name="6 Dispute reasons", index=False)
    for code, table in cutoff_tables.items():
        table.to_excel(writer, sheet_name=f"7 Cut-off {code}")
print(f"\nsaved {OUTPUT_FILE}: {os.path.getsize(OUTPUT_FILE) / 1024:.0f} KB")